# Práctica — Clase 8: perros vs. gatos

Completa i due esercizi. Ognuno ha celle con `# TODO:` e una soluzione nascosta sotto.

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

TAMANO_IMAGEN = (128, 128)

(dataset_entreno, dataset_validacion), info = tfds.load(
    "cats_vs_dogs",
    split=["train[:80%]", "train[80%:]"],
    as_supervised=True,
    with_info=True,
)

def preprocesar(imagen, etiqueta):
    imagen = tf.image.resize(imagen, TAMANO_IMAGEN) / 255.0
    return imagen, etiqueta

dataset_entreno = dataset_entreno.map(preprocesar).batch(32).prefetch(1)
dataset_validacion = dataset_validacion.map(preprocesar).batch(32).prefetch(1)

## Esercizio 1 — Completa l'aumento dei dati

**Enunciato**: definisci `aumento_datos` con tre strati: flip orizzontale casuale, rotazione casuale di 0.1, e zoom casuale di 0.1.

**Suggerimento**: gli strati si chiamano `RandomFlip`, `RandomRotation` e `RandomZoom`.


In [ ]:
aumento_datos = tf.keras.Sequential([
    # TODO: añade RandomFlip("horizontal")
    # TODO: añade RandomRotation(0.1)
    # TODO: añade RandomZoom(0.1)
])

modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 128, 3)),
    aumento_datos,
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.fit(dataset_entreno, epochs=3, validation_data=dataset_validacion)

<details>
<summary>💡 Vedi soluzione</summary>

```python
aumento_datos = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])
```
</details>

## Esercizio 2 — Conta i successi e gli errori del modello

**Enunciato**: completa la funzione `contar_aciertos` affinché confronti, lotto per lotto del set di validazione, la predizione del modello (soglia 0.5) con l'etichetta reale, e restituisca il numero totale di successi e di errori.

**Suggerimento**: una predizione è corretta quando `(predicción >= 0.5)` coincide con l'etichetta reale (0 o 1).


In [ ]:
import numpy as np

def contar_aciertos(modelo, dataset):
    aciertos = 0
    fallos = 0
    for imagenes, etiquetas in dataset:
        predicciones = modelo.predict(imagenes, verbose=0).flatten()
        predicciones_binarias = (predicciones >= 0.5).astype(int)
        # TODO: suma a 'aciertos' los casos donde predicciones_binarias coincide con etiquetas
        # TODO: suma a 'fallos' los casos donde no coincide
    return aciertos, fallos

aciertos, fallos = contar_aciertos(modelo, dataset_validacion)
print(f"Aciertos: {aciertos}, Fallos: {fallos}")

<details>
<summary>💡 Vedi soluzione</summary>

```python
def contar_aciertos(modelo, dataset):
    aciertos = 0
    fallos = 0
    for imagenes, etiquetas in dataset:
        predicciones = modelo.predict(imagenes, verbose=0).flatten()
        predicciones_binarias = (predicciones >= 0.5).astype(int)
        coincidencias = (predicciones_binarias == etiquetas.numpy())
        aciertos += int(np.sum(coincidencias))
        fallos += int(np.sum(~coincidencias))
    return aciertos, fallos
```
</details>